# Zarr datasets and dataloaders

Create a tiny local Zarr store, build a dataset/dataloader, then read the same store from a read-only zip archive.


In [ ]:
function find_repo_root(start = pwd())
    dir = abspath(start)
    while !isfile(joinpath(dir, "Project.toml")) || !isdir(joinpath(dir, "src"))
        parent = dirname(dir)
        parent == dir && error("Could not find QuantumGraph.jl repository root")
        dir = parent
    end
    dir
end

repo_root = find_repo_root()
cd(repo_root)
using Pkg
Pkg.activate(repo_root)


In [ ]:
using QuantumGraph
using MLUtils
using Zarr
using ZipFile


In [ ]:
function create_array(group, name, values; chunks = size(values))
    array = Zarr.zcreate(eltype(values), group, name, size(values)...; chunks)
    array[:] = values
    array
end

store_path = joinpath(mktempdir(), "graphs.zarr")
root = Zarr.zgroup(store_path)
create_array(root, "num_samples", [3])
create_array(root, "adjacency_matrix", reshape(Float32[0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0], 3, 2, 2))
create_array(root, "dimension", Float32[2, 2, 2])
create_array(root, "atomcount", Float32[2, 2, 2])


In [ ]:
store = open_zarr_for_dataset(store_path; required_arrays = ["adjacency_matrix", "num_samples"])
dataset = construct_dataset(store_path; required_arrays = ["adjacency_matrix"])
length(dataset), dataset[1]


In [ ]:
loader = dataset_dataloader(dataset; batchsize = 2, collate = false, shuffle = false)
first(loader)


In [ ]:
function zip_directory(source_dir, zip_path)
    zip = ZipFile.Writer(zip_path)
    try
        for (root_dir, _, files) in walkdir(source_dir)
            for file in files
                full_path = joinpath(root_dir, file)
                rel_path = relpath(full_path, dirname(source_dir))
                writer = ZipFile.addfile(zip, rel_path)
                write(writer, read(full_path))
            end
        end
    finally
        close(zip)
    end
    zip_path
end

zip_path = zip_directory(store_path, joinpath(mktempdir(), "graphs.zarr.zip"))
zip_dataset = construct_dataset(zip_path; required_arrays = ["adjacency_matrix"])
length(zip_dataset), zip_dataset[1].source
